# Числовые и категориальные признаки

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Объяснять выбор масштаба чисел и кодирования категорий без ложного порядка.

## Числовая запись не всегда число по смыслу

Коды районов 1,2,3 обозначают категории. Вычитание 3−1 не задаёт полезного расстояния между районами. **One-hot кодирование** создаёт по столбцу на категорию: для А и Б получаются [1,0] и [0,1]. Число 1 указывает принадлежность, а не величину признака.

Числовые измерения часто приводят к сравнимому масштабу. StandardScaler вычитает обучающее среднее и делит на обучающее стандартное отклонение. Для [10,14] получаем среднее 12, стандартное отклонение 2 и значения [−1,1]. Постоянный обучающий столбец обрабатывается без деления на ноль. Масштабирование особенно важно для расстояний и штрафов на коэффициенты.

## Разобранный пример: неизвестная категория

Обучение видело бумагу А и Б, новый объект имеет В. OneHotEncoder с handle_unknown='ignore' выдаст нули обоих известных столбцов. Это позволяет выполнить преобразование, но не означает, что модель знает свойства В. Политика неизвестных категорий должна быть частью решения.

Порядковые категории, например «малый, средний, большой», иногда допускают числовой порядок, но разность кодов всё равно не гарантирует равенства реальных промежутков. Решение зависит от смысла данных и модели. Кодирование по среднему ответа особенно опасно: без вычисления внутри обучающих блоков оно передаст сведения о правильных ответах в признаки.

В ноутбуке StandardScaler и OneHotEncoder сначала обучаются через fit, затем применяются через transform. Параметр sparse_output=False просит обычную плотную таблицу: все ячейки, включая нули, занимают место. При большом числе категорий заранее оцените память.

Почему опасна замена категории средним ответом, или **target encoding**? Если категория встретилась один раз с ответом 7, её среднее тоже 7: признак фактически скопировал собственный ответ. Для обучающих строк такие средние считают по другим внутренним блокам, исключая ответ самой строки. Для validation используют только разрешённые обучающие данные. Обычное one-hot-кодирование не использует y и такой проблемы не создаёт.

Сборка `ColumnTransformer` и `Pipeline` уже разобрана в [уроке Python](../../../python/ml-tools/pipelines/). Здесь важно другое: какое различие между объектами создаёт выбранное кодирование и что произойдёт с неизвестной категорией. Код повторяет небольшой конвейер, чтобы ноутбук оставался самостоятельным.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
scale = StandardScaler().fit([[10.], [14.]])
assert np.allclose(scale.transform([[16.]]), [[2.]])
encode = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit([["А"], ["Б"]])
assert encode.transform([["В"]]).tolist() == [[0., 0.]]
print(encode.categories_)  # выученные категории и их порядок


## Типичные ошибки

Не создавайте числовой порядок номинальных категорий случайными кодами. Не пересчитывайте масштаб отдельно на тесте.

## Задачи

### 1. Базовая

Почему код района 3 не обязательно больше района 1 по смыслу?

<details>
<summary>Подсказка</summary>

Это имя категории.

</details>



### 2. Базовая

Где сохраняются среднее и масштаб?

<details>
<summary>Подсказка</summary>

Они должны повторно применяться.

</details>



### 3. Прикладная

Стандартизируйте 16 при среднем 12 и масштабе 2.

<details>
<summary>Подсказка</summary>

Вычтите и разделите.

</details>



### 4. Прикладная

Закодируйте Б при столбцах А,Б,В.

<details>
<summary>Подсказка</summary>

Единица только у своей категории.

</details>



### 5. Олимпиадная
Неизвестную категорию закодировали нулевым вектором, известные А и Б — векторами (1,0) и (0,1). Доказывает ли нулевой код, что категория не влияет на KNN? Сравните расстояния до А и Б с расстояниями от кода А.

<details>
<summary>Подсказка</summary>

Нулевой вектор — тоже точка в пространстве, а не команда пропустить признак.

</details>



### 6. Олимпиадная

Почему кодирование средним ответа по всей таблице создаёт утечку?

<details>
<summary>Подсказка</summary>

Ответ строки участвует в её же признаке.

</details>



## Итоги и самопроверка

Сравните смысл измерения и идентификатора. Как преобразуется неизвестная категория и что это говорит об ограничениях модели?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Числа могут быть только обозначениями и не задавать количественный порядок.

### Задача 2

В состоянии преобразователя после fit на train.

### Задача 3

(16−12)/2=2.

### Задача 4

[0,1,0].

### Задача 5

От (0,0) до обоих кодов расстояние 1. От кода А=(1,0) до А расстояние 0, до Б — √2. Состав ближайших соседей и ничьи могут измениться. Нулевой код означает конкретное представление неизвестной категории; он не доказывает ни отсутствия влияния на прогноз, ни надёжности прогноза для ранее не встречавшихся категорий.

### Задача 6

Проверочные ответы влияют на преобразование. Нужна схема обучения такого кодирования только по допустимым обучающим объектам.